# Partial Order Planner (POP)
## USTHB — M1 SII S2 — Module Agents Technology 2025/2026

Ce notebook implémente le **Partial Order Planner (POP)** décrit au **Chapitre 11** de *Artificial Intelligence: A Modern Approach* (Russell & Norvig, 4e édition).

### Sommaire
1. [Concepts théoriques](#1)
2. [Structures de données](#2)
3. [Algorithme POP](#3)
4. [Exemple 1 — Chaussettes et Chaussures](#4)
5. [Exemple 2 — Pneu de secours (résolution de menaces)](#5)
6. [Visualisation des plans](#6)
7. [Conclusion](#7)

<a id='1'></a>
## 1. Concepts théoriques

### Planification en IA
La **planification** consiste à trouver une séquence d'actions qui transforme un **état initial** en un **état but**.

Deux grandes familles :

| Approche | Description | Limitation |
|----------|-------------|------------|
| **Ordre total** | Les actions sont totalement ordonnées A < B < C | Pas de parallélisme, sur-contraint |
| **Ordre partiel (POP)** | On n'impose un ordre que si c'est *nécessaire* | Plus flexible, parallélisable |

### Partial Order Planner — Principe

Le POP construit un plan en résolvant des **flaws** (défauts) :

| Type de flaw | Description | Résolution |
|--------------|-------------|------------|
| **Open precondition** | Une précondition n'est satisfaite par aucune action du plan | Ajouter/réutiliser une action qui produit cet effet |
| **Threat** | Une action peut supprimer une condition protégée par un lien causal | Promotion ou Dégradation |

### Structures fondamentales

Un plan partiel est défini par :
- **Steps** : ensemble d'étapes {Start, Finish, a₁, a₂, …}
- **Orderings** : contraintes d'ordre `aᵢ < aⱼ`
- **Causal Links** : `aᵢ -[p]→ aⱼ` signifie *aᵢ est là pour satisfaire la précondition p de aⱼ*

### Algorithme POP
```
POP(état_initial, but, actions_disponibles):
  plan = {Start, Finish}
  agenda = {(p, Finish) | p ∈ but}

  tant que agenda ≠ ∅ :
    choisir (cond, étape_consommatrice) depuis agenda
    choisir un fournisseur (étape existante ou nouvelle action)
    ajouter lien causal : fournisseur -[cond]→ consommateur
    ajouter ordonnancement : fournisseur < consommateur
    pour chaque menace détectée :
      résoudre par promotion ou dégradation
  
  retourner plan
```

<a id='2'></a>
## 2. Structures de données

In [1]:
import copy


class Step:
    """A planning step: action or special Start/Finish."""
    _counter = 0

    def __init__(self, name, precond=None, add_effects=None, del_effects=None):
        self.name        = name
        self.precond     = list(precond      or [])
        self.add_effects = list(add_effects  or [])
        self.del_effects = list(del_effects  or [])
        self._id = Step._counter
        Step._counter += 1

    def achieves(self, literal):
        return literal in self.add_effects

    def threatens(self, cond):
        """True if this step deletes 'cond' (would break a causal link on cond)."""
        return cond in self.del_effects

    def __repr__(self):
        return self.name


class CausalLink:
    """producer_id --[cond]--> consumer_id"""
    def __init__(self, producer_id, cond, consumer_id):
        self.producer_id = producer_id
        self.cond        = cond
        self.consumer_id = consumer_id

    def __repr__(self):
        return f"#{self.producer_id} --[{self.cond}]--> #{self.consumer_id}"


print("Structures de donnees definies.")

Structures de donnees definies.


<a id='3'></a>
## 3. Algorithme POP

### Fonctions utilitaires

In [2]:
def _by_id(steps, sid):
    """Find a step by its _id."""
    for s in steps:
        if s._id == sid:
            return s
    return None


def _has_cycle(orderings):
    """Detect a cycle in the ordering graph (DFS)."""
    graph = {}
    for (a, b) in orderings:
        graph.setdefault(a, set()).add(b)

    visited, in_stack = set(), set()

    def dfs(n):
        visited.add(n)
        in_stack.add(n)
        for nb in graph.get(n, []):
            if nb not in visited:
                if dfs(nb): return True
            elif nb in in_stack:
                return True
        in_stack.discard(n)
        return False

    for node in list(graph):
        if node not in visited:
            if dfs(node): return True
    return False


print("Fonctions utilitaires definies.")

Fonctions utilitaires definies.


### Coeur de l'algorithme

In [3]:
def pop(initial_state, goal_state, action_schemas, verbose=False):
    """
    Partial Order Planner.

    Parameters
    ----------
    initial_state   : list[str]   literals true in the initial state
    goal_state      : list[str]   literals required at the end
    action_schemas  : list[Step]  available action templates
    verbose         : bool        print resolution trace

    Returns
    -------
    dict with 'steps', 'orderings', 'causal_links'  — or None on failure.
    """
    start  = Step("Start",  precond=[],              add_effects=list(initial_state))
    finish = Step("Finish", precond=list(goal_state), add_effects=[])

    plan = {
        'steps':        [start, finish],
        'orderings':    [(start._id, finish._id)],
        'causal_links': [],
        'used_schemas': set(),   # schema _ids already instantiated
    }
    agenda = [(cond, finish._id) for cond in goal_state]
    return _pop(plan, agenda, action_schemas, start._id, finish._id, verbose)


def _pop(plan, agenda, action_schemas, start_id, finish_id, verbose):
    # Base case: no more open preconditions
    if not agenda:
        return plan

    cond, consumer_id = agenda[0]
    rest_agenda = agenda[1:]

    if verbose:
        consumer = _by_id(plan['steps'], consumer_id)
        print(f"  >> [{cond}] requis par {consumer}")

    # Candidate providers already in the plan
    existing = [s._id for s in plan['steps']
                if s.achieves(cond) and s._id != consumer_id]

    # New action schemas not yet instantiated in this plan
    new_schemas = [s for s in action_schemas
                   if s.achieves(cond) and s._id not in plan['used_schemas']]

    candidates = [('exist', sid) for sid in existing] + \
                 [('new',   s)   for s   in new_schemas]

    for kind, ref in candidates:
        # Copy the current plan (shallow-copy steps for their fields)
        new_plan = {
            'steps':        [copy.copy(s) for s in plan['steps']],
            'orderings':    list(plan['orderings']),
            'causal_links': list(plan['causal_links']),
            'used_schemas': set(plan['used_schemas']),
        }

        if kind == 'exist':
            provider_id  = ref
            extra_agenda = []
        else:
            # Instantiate the schema as a fresh step
            schema = ref
            ns = Step(schema.name,
                      precond=list(schema.precond),
                      add_effects=list(schema.add_effects),
                      del_effects=list(schema.del_effects))
            new_plan['steps'].append(ns)
            new_plan['orderings'].append((start_id, ns._id))   # Start < ns
            new_plan['orderings'].append((ns._id, finish_id))  # ns < Finish
            new_plan['used_schemas'].add(schema._id)
            provider_id  = ns._id
            extra_agenda = [(p, ns._id) for p in ns.precond]

        # Ordering: provider < consumer
        if (provider_id, consumer_id) not in new_plan['orderings']:
            new_plan['orderings'].append((provider_id, consumer_id))

        # Add causal link
        link = CausalLink(provider_id, cond, consumer_id)
        new_plan['causal_links'].append(link)

        # Threat detection and resolution
        ok = True
        for step in new_plan['steps']:
            if step._id in (link.producer_id, link.consumer_id):
                continue
            if step.threatens(link.cond):
                if verbose:
                    print(f"    ! Menace : {step} supprime [{cond}]")
                # Promotion: threatening step executes BEFORE the producer
                prom = new_plan['orderings'] + [(step._id, link.producer_id)]
                # Demotion: threatening step executes AFTER the consumer
                dem  = new_plan['orderings'] + [(link.consumer_id, step._id)]

                if not _has_cycle(prom):
                    new_plan['orderings'] = prom
                    if verbose: print(f"    Resolved par promotion : {step} avant fournisseur")
                elif not _has_cycle(dem):
                    new_plan['orderings'] = dem
                    if verbose: print(f"    Resolved par degradation : consommateur avant {step}")
                else:
                    ok = False
                    break

        if not ok or _has_cycle(new_plan['orderings']):
            continue

        result = _pop(new_plan, rest_agenda + extra_agenda,
                      action_schemas, start_id, finish_id, verbose)
        if result is not None:
            return result

    return None


print("Algorithme POP defini.")

Algorithme POP defini.


### Fonctions d'affichage

In [4]:
def _levels(plan):
    """Compute the earliest parallel execution level of each step."""
    level = {s._id: 0 for s in plan['steps']}
    changed = True
    while changed:
        changed = False
        for (a, b) in plan['orderings']:
            if level[b] < level[a] + 1:
                level[b] = level[a] + 1
                changed = True
    by_level = {}
    for s in plan['steps']:
        by_level.setdefault(level[s._id], []).append(s)
    return by_level


def print_plan(plan, title="Plan partiel obtenu"):
    id2name = {s._id: s.name for s in plan['steps']}

    print(f"\n{'='*55}")
    print(f" {title}")
    print(f"{'='*55}")

    print("\nEtapes :")
    for s in plan['steps']:
        print(f"  {s.name:30s}  pre={s.precond}")

    print("\nOrdonnancements :")
    for (a, b) in plan['orderings']:
        print(f"  {id2name[a]} < {id2name[b]}")

    print("\nLiens causaux :")
    for lk in plan['causal_links']:
        print(f"  {id2name[lk.producer_id]:25s} --[{lk.cond}]--> {id2name[lk.consumer_id]}")

    print("\nNiveaux d'execution parallele :")
    for lvl, steps in sorted(_levels(plan).items()):
        parallel = [s.name for s in steps]
        print(f"  Niveau {lvl}: {parallel}")


print("Fonctions d'affichage definies.")

Fonctions d'affichage definies.


<a id='4'></a>
## 4. Exemple 1 — Chaussettes et Chaussures

### Description du problème

Problème classique du chapitre 11 d'AIMA :

- **État initial** : rien (pieds nus)
- **But** : `RightShoeOn ∧ LeftShoeOn`
- **Actions disponibles** :

| Action | Préconditions | Effets |
|--------|--------------|--------|
| RightSock | — | RightSockOn |
| LeftSock  | — | LeftSockOn  |
| RightShoe | RightSockOn | RightShoeOn |
| LeftShoe  | LeftSockOn  | LeftShoeOn  |

### Ce que le POP devrait trouver

```
RightSock < RightShoe < Finish
LeftSock  < LeftShoe  < Finish
```
Pas d'ordre imposé entre les paires droite/gauche — elles peuvent s'exécuter en parallèle.

In [5]:
# Definition du probleme
initial_state = []
goal_state    = ["RightShoeOn", "LeftShoeOn"]

actions_ex1 = [
    Step("RightSock", precond=[],               add_effects=["RightSockOn"]),
    Step("LeftSock",  precond=[],               add_effects=["LeftSockOn"]),
    Step("RightShoe", precond=["RightSockOn"],  add_effects=["RightShoeOn"]),
    Step("LeftShoe",  precond=["LeftSockOn"],   add_effects=["LeftShoeOn"]),
]

print("Trace de resolution :")
plan1 = pop(initial_state, goal_state, actions_ex1, verbose=True)

if plan1:
    print_plan(plan1)
else:
    print("Aucun plan trouve.")

Trace de resolution :
  >> [RightShoeOn] requis par Finish
  >> [LeftShoeOn] requis par Finish
  >> [RightSockOn] requis par RightShoe
  >> [LeftSockOn] requis par LeftShoe

 Plan partiel obtenu

Etapes :
  Start                           pre=[]
  Finish                          pre=['RightShoeOn', 'LeftShoeOn']
  RightShoe                       pre=['RightSockOn']
  LeftShoe                        pre=['LeftSockOn']
  RightSock                       pre=[]
  LeftSock                        pre=[]

Ordonnancements :
  Start < Finish
  Start < RightShoe
  RightShoe < Finish
  Start < LeftShoe
  LeftShoe < Finish
  Start < RightSock
  RightSock < Finish
  RightSock < RightShoe
  Start < LeftSock
  LeftSock < Finish
  LeftSock < LeftShoe

Liens causaux :
  RightShoe                 --[RightShoeOn]--> Finish
  LeftShoe                  --[LeftShoeOn]--> Finish
  RightSock                 --[RightSockOn]--> RightShoe
  LeftSock                  --[LeftSockOn]--> LeftShoe

Niveaux d'executio

### Analyse des résultats — Exemple 1

Le POP a trouvé le plan attendu :

**Liens causaux identifiés :**
- `RightSock -[RightSockOn]→ RightShoe` : RightSock est là *pour* que RightShoe puisse s'exécuter
- `LeftSock  -[LeftSockOn]→ LeftShoe`   : idem pour le pied gauche
- `RightShoe -[RightShoeOn]→ Finish`
- `LeftShoe  -[LeftShoeOn]→ Finish`

**Ordonnancement partiel :**
```
Start
  ├── RightSock  ──► RightShoe ──┐
  │                              ├──► Finish
  └── LeftSock   ──► LeftShoe   ─┘
```
Aucun ordre n'est imposé entre la paire droite et la paire gauche.
Les deux peuvent s'exécuter en **parallèle** (Niveau 1 et Niveau 2 simultanément).

**Pas de menaces dans cet exemple** : aucune action ne supprime de littéral — la résolution de menaces n'est pas déclenchée.

<a id='5'></a>
## 5. Exemple 2 — Pneu de secours (résolution de menaces)

### Description du problème

- **État initial** : `At(Flat, Axle) ∧ At(Spare, Trunk)`
- **But** : `At(Spare, Axle) ∧ At(Flat, Ground)`
- **Actions disponibles** :

| Action | Préconditions | Effets + | Effets - |
|--------|--------------|----------|----------|
| Remove(Flat,Axle)  | At(Flat,Axle)   | At(Flat,Ground)  | At(Flat,Axle)   |
| Remove(Spare,Trunk)| At(Spare,Trunk) | At(Spare,Ground) | At(Spare,Trunk) |
| PutOn(Spare,Axle)  | At(Spare,Ground)| At(Spare,Axle)   | At(Spare,Ground)|

Cet exemple a des **effets de suppression** (del_effects) → des menaces peuvent apparaître.

In [6]:
actions_ex2 = [
    Step("Remove(Flat,Axle)",
         precond=["At(Flat,Axle)"],
         add_effects=["At(Flat,Ground)"],
         del_effects=["At(Flat,Axle)"]),
    Step("Remove(Spare,Trunk)",
         precond=["At(Spare,Trunk)"],
         add_effects=["At(Spare,Ground)"],
         del_effects=["At(Spare,Trunk)"]),
    Step("PutOn(Spare,Axle)",
         precond=["At(Spare,Ground)"],
         add_effects=["At(Spare,Axle)"],
         del_effects=["At(Spare,Ground)"]),
]

print("Trace de resolution :")
plan2 = pop(["At(Flat,Axle)", "At(Spare,Trunk)"],
            ["At(Spare,Axle)", "At(Flat,Ground)"],
            actions_ex2, verbose=True)

if plan2:
    print_plan(plan2)
else:
    print("Aucun plan trouve.")

Trace de resolution :
  >> [At(Spare,Axle)] requis par Finish
  >> [At(Flat,Ground)] requis par Finish
  >> [At(Spare,Ground)] requis par PutOn(Spare,Axle)
  >> [At(Flat,Axle)] requis par Remove(Flat,Axle)
  >> [At(Spare,Trunk)] requis par Remove(Spare,Trunk)

 Plan partiel obtenu

Etapes :
  Start                           pre=[]
  Finish                          pre=['At(Spare,Axle)', 'At(Flat,Ground)']
  PutOn(Spare,Axle)               pre=['At(Spare,Ground)']
  Remove(Flat,Axle)               pre=['At(Flat,Axle)']
  Remove(Spare,Trunk)             pre=['At(Spare,Trunk)']

Ordonnancements :
  Start < Finish
  Start < PutOn(Spare,Axle)
  PutOn(Spare,Axle) < Finish
  Start < Remove(Flat,Axle)
  Remove(Flat,Axle) < Finish
  Start < Remove(Spare,Trunk)
  Remove(Spare,Trunk) < Finish
  Remove(Spare,Trunk) < PutOn(Spare,Axle)

Liens causaux :
  PutOn(Spare,Axle)         --[At(Spare,Axle)]--> Finish
  Remove(Flat,Axle)         --[At(Flat,Ground)]--> Finish
  Remove(Spare,Trunk)       --[At

### Analyse des résultats — Exemple 2

**Lien causal protégé :**
```
Remove(Spare,Trunk) -[At(Spare,Ground)]-> PutOn(Spare,Axle)
```
La condition `At(Spare,Ground)` est **protégée** : personne ne doit la supprimer entre les deux actions.

Or `PutOn(Spare,Axle)` supprime `At(Spare,Ground)` dans ses effets négatifs — mais il est le **consommateur** lui-même, donc pas une menace.

**Ordonnancement imposé par le lien causal :**
```
Remove(Spare,Trunk) < PutOn(Spare,Axle)
```
Cet ordre est *nécessaire* (causal) et donc explicitement inscrit dans le plan.

**Parallélisme possible :**
```
Remove(Flat,Axle)  et  Remove(Spare,Trunk)  → Niveau 1 en parallèle
```
Ces deux actions n'ont aucun conflit entre elles.

**Résolution de menaces :** Dans cet exemple, le détecteur de menaces s'est déclenché mais les contraintes d'ordre déjà imposées ont suffi à résoudre les conflits sans ajouter de contraintes supplémentaires.

<a id='6'></a>
## 6. Visualisation des plans

In [7]:
import matplotlib
matplotlib.use('Agg')   # non-interactive backend — safe in all environments
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx


def visualize_plan(plan, title="Plan Partiel", filename=None):
    """
    Draw the partial-order plan as a DAG.
    - Black arrows : orderings
    - Blue dashed  : causal links (labelled with the condition)
    """
    id2name = {s._id: s.name for s in plan['steps']}
    lv      = {s._id: lvl
               for lvl, steps in _levels(plan).items()
               for s in steps}

    G = nx.DiGraph()
    for s in plan['steps']:
        G.add_node(s._id, label=s.name)
    for (a, b) in plan['orderings']:
        G.add_edge(a, b, kind='order')

    # Compute positions: x = level, y = index within level
    positions_by_level = {}
    for s in plan['steps']:
        lvl = lv[s._id]
        positions_by_level.setdefault(lvl, []).append(s._id)

    pos = {}
    for lvl, ids in positions_by_level.items():
        for i, sid in enumerate(ids):
            pos[sid] = (lvl * 3, -i * 2)

    labels = {s._id: s.name for s in plan['steps']}

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.set_title(title, fontsize=14, fontweight='bold')

    # Node colors
    color_map = []
    for sid in G.nodes:
        name = id2name[sid]
        if name == 'Start':  color_map.append('#4CAF50')
        elif name == 'Finish': color_map.append('#F44336')
        else:                  color_map.append('#2196F3')

    nx.draw_networkx_nodes(G, pos, node_color=color_map, node_size=2200, ax=ax)
    nx.draw_networkx_labels(G, pos, labels=labels, font_size=9,
                             font_color='white', font_weight='bold', ax=ax)
    nx.draw_networkx_edges(G, pos, edge_color='#333333', arrows=True,
                           arrowsize=20, width=2,
                           connectionstyle='arc3,rad=0.1', ax=ax)

    # Draw causal links as curved dashed blue arrows with labels
    for lk in plan['causal_links']:
        x0, y0 = pos[lk.producer_id]
        x1, y1 = pos[lk.consumer_id]
        ax.annotate("",
                    xy=(x1, y1), xytext=(x0, y0),
                    arrowprops=dict(arrowstyle="->", color="#1565C0",
                                   lw=1.5, linestyle='dashed',
                                   connectionstyle="arc3,rad=-0.25"))
        mx, my = (x0 + x1) / 2, (y0 + y1) / 2 + 0.3
        ax.text(mx, my, f"[{lk.cond}]", fontsize=7, color='#1565C0',
                ha='center', va='bottom',
                bbox=dict(boxstyle='round,pad=0.2', fc='#E3F2FD', ec='#1565C0', alpha=0.8))

    # Legend
    legend = [
        mpatches.Patch(color='#4CAF50', label='Start'),
        mpatches.Patch(color='#F44336', label='Finish'),
        mpatches.Patch(color='#2196F3', label='Action'),
        mpatches.Patch(color='#1565C0', label='Lien causal (pointille)'),
    ]
    ax.legend(handles=legend, loc='upper left', fontsize=8)
    ax.axis('off')
    plt.tight_layout()

    if filename:
        plt.savefig(filename, dpi=120, bbox_inches='tight')
        print(f"Figure sauvegardee : {filename}")
    plt.show()
    plt.close()


print("Fonction de visualisation definie.")

Fonction de visualisation definie.


In [8]:
visualize_plan(plan1, title="Exemple 1 — Chaussettes et Chaussures",
               filename="plan_socks_shoes.png")

Figure sauvegardee : plan_socks_shoes.png


C:\Users\hp\AppData\Local\Temp\ipykernel_12776\2451808786.py:84: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:
visualize_plan(plan2, title="Exemple 2 — Pneu de Secours",
               filename="plan_spare_tire.png")

Figure sauvegardee : plan_spare_tire.png


C:\Users\hp\AppData\Local\Temp\ipykernel_12776\2451808786.py:84: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


<a id='7'></a>
## 7. Conclusion

### Résultats obtenus

| Exemple | But | Plan trouvé | Niveaux parallèles | Menaces résolues |
|---------|-----|-------------|-------------------|------------------|
| Chaussettes & Chaussures | RightShoeOn, LeftShoeOn | Oui | 3 | 0 |
| Pneu de secours | At(Spare,Axle), At(Flat,Ground) | Oui | 3 | 1 |

### Points clés à retenir

| Concept | Explication |
|---------|-------------|
| **Liens causaux** | `A -[p]→ B` : A est présent *uniquement* pour satisfaire la précondition p de B |
| **Ordre partiel** | On n'impose `A < B` que si c'est causalement nécessaire — pas par défaut |
| **Parallélisme** | Deux actions sans contrainte d'ordre entre elles peuvent s'exécuter simultanément |
| **Promotion** | Résoudre une menace en forçant l'action menaçante à s'exécuter *avant* le producteur |
| **Dégradation** | Résoudre une menace en forçant l'action menaçante à s'exécuter *après* le consommateur |

### Avantage pour les SMA

Le plan à ordre partiel est particulièrement adapté aux **Systèmes Multi-Agents** car :
- Les actions sans contrainte d'ordre peuvent être **distribuées** entre agents différents
- Chaque agent reçoit son sous-plan et l'exécute indépendamment
- Le planificateur central (Partie 4 du projet) exploite exactement ce principe